# Trabajo Copa Mundial
Este notebook resuelve íntegramente el proyecto sobre datos de la Copa Mundial, aplicando PySpark, RDDs, DataFrames y Spark SQL.

## Parte 1: Configuración Inicial del Entorno
1.1. Instalación en entorno de ejecución.
1.2. Variables de entorno.
1.3. Creación de SparkSession 'MundialAnalysis' y SparkContext.

In [ ]:
# 1.1 Instalación de Java y Apache Spark (Ideal para Google Colab)
!apt-get update -qq > /dev/null
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark


In [ ]:
import os
import sys
import findspark

# 1.2 Variables de Entorno (Ajustado para el entorno de Google Colab)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pyspark.sql.functions as F

# 1.3 SparkSession y SparkContext
spark = SparkSession.builder \
    .appName("MundialAnalysis") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

sc = spark.sparkContext
print("SparkSession 'MundialAnalysis' creada y contexto iniciado de forma exitosa.")


## Parte 2: RDDs - Creación y Unión

In [ ]:
# NOTA: Asegúrese de subir los archivos CSV a la carpeta '../data/' o ajuste este path a './'
path_jugadores = "../data/jugadores.csv"

# 2.1 y 2.2 Crear RDDs llamados jugador1 y jugador2 con 6 particiones
jugador1 = sc.textFile(path_jugadores, minPartitions=6)
jugador2 = sc.textFile(path_jugadores, minPartitions=6)

# 2.3 Crear RDD jugadorTotal con la unión
jugadorTotal_bruto = jugador1.union(jugador2)
header = jugadorTotal_bruto.first()
jugadorTotal = jugadorTotal_bruto.filter(lambda row: row != header)

# 2.4 Mostrar cantidad de registros
print("Cantidad total de registros unidos en jugadorTotal:", jugadorTotal.count())

# 2.5 Convertir RDD en DataFrame 'jugadores'
def parse_jugador(line):
    cols = line.split(',')
    return (int(cols[0]), cols[1], cols[2], int(cols[3]), int(cols[4]), int(cols[5]), cols[6], int(cols[7]))

rdd_parsed = jugadorTotal.map(parse_jugador)

schema_jugadores = StructType([
    StructField("jugador_id", IntegerType(), True),
    StructField("nombre", StringType(), True),  # Cumplimiento estricto
    StructField("apellido", StringType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", IntegerType(), True),
    StructField("peso", IntegerType(), True),
    StructField("posicion", StringType(), True),
    StructField("equipo_id", IntegerType(), True)
])

jugadores = spark.createDataFrame(rdd_parsed, schema_jugadores)

# 2.6 Mostrar el esquema final
print("\n--- Esquema y Muestra del DataFrame 'jugadores' (2.6) ---")
jugadores.printSchema()
jugadores.show(5)


## Parte 3: RDDs - Transformaciones

In [ ]:
# 3.1 Filtro por Edad > 30
MayorEdad = jugadorTotal.filter(lambda x: int(x.split(',')[3]) > 30)

# 3.2 Filtro por Posición Defensa
Jugadores_Defensa = jugadorTotal.filter(lambda x: x.split(',')[6].strip() == 'Defensa')

# 3.3 A mayúsculas (nombre y apellido)
def to_upper_names(line):
    cols = line.split(',')
    cols[1] = cols[1].upper()
    cols[2] = cols[2].upper()
    return ','.join(cols)
jugadorTotal_mayusculas = jugadorTotal.map(to_upper_names)

print("Muestra 3 registros de MayorEdad:")
for r in MayorEdad.take(3): print(r)

print("\nMuestra 3 registros de Jugadores_Defensa:")
for r in Jugadores_Defensa.take(3): print(r)

print("\nMuestra 3 registros de conversión a Mayúsculas:")
for r in jugadorTotal_mayusculas.take(3): print(r)


## Parte 4: DataFrames - Creación e Integración

In [ ]:
# 4.1 Crear DataFrames restantes
equipos = spark.read.csv("../data/equipos.csv", header=True, inferSchema=True)
partidos = spark.read.csv("../data/partidos.csv", header=True, inferSchema=True)
estadios = spark.read.csv("../data/estadios.csv", header=True, inferSchema=True)

# Lectura robusta con multiline=True para soportar torneos.json de forma segura
torneos = spark.read.option("multiline", "true").json("../data/torneos.json")

# Evitando colisiones y ambigüedades lógicas en las uniones:
equipos = equipos.withColumnRenamed("nombre", "nombre_equipo")
estadios = estadios.withColumnRenamed("nombre", "nombre_estadio") \
                   .withColumnRenamed("pais", "pais_estadio")
torneos = torneos.withColumnRenamed("nombre", "nombre_torneo")

# 4.2 Optimización obligatoria
jugadores.cache()
equipos.cache()
partidos.cache()
estadios.cache()
torneos.cache()

# 4.3 Creado Dataframe `mundial_completo` mediante operaciones de JOIN iterativas
condicion_partidos = (F.col("equipo_id") == F.col("equipo_local_id")) | \
                     (F.col("equipo_id") == F.col("equipo_visitante_id"))

mundial_completo_no_partition = jugadores.join(equipos, "equipo_id", "inner") \
    .join(partidos, condicion_partidos, "left") \
    .join(estadios, "estadio_id", "left") \
    .join(torneos, "torneo_id", "left")


## Parte 5: Paralelismo y Parte 6: Inspección

In [ ]:
# 5.1 Paralelizar a 5 particiones
mundial_completo = mundial_completo_no_partition.repartition(5)

# 5.2 Verificar número de particiones
print(f"Verificación de particiones (deberían ser 5): {mundial_completo.rdd.getNumPartitions()}")

# 6.1 Mostrar la cantidad de filas, tipo de datos y el esquema
print(f"\nConteo Total de Filas extraídas por cruce general: {mundial_completo.count()}")
print("\n--- Esquema Final (Tipos de Datos) ---")
mundial_completo.printSchema()
mundial_completo.show(5)


## Parte 7: Columnas Calculadas

In [ ]:
mundial_completo = mundial_completo \
    .withColumn("IMC", F.col("peso") / F.pow(F.col("altura") / 100.0, 2)) \
    .withColumn("Categoria_Edad",
        F.when(F.col("edad") < 25, "Joven")
         .when((F.col("edad") >= 25) & (F.col("edad") <= 32), "Experimentado")
         .otherwise("Veterano")
    ) \
    .withColumn("Resultado_Partido",
        F.when(F.col("goles_local") > F.col("goles_visitante"), "Victoria Local")
         .when(F.col("goles_visitante") > F.col("goles_local"), "Victoria Visitante")
         .otherwise("Empate")
    )

print("Muestra de las nuevas columnas calculadas (IMC, Categoria_Edad, Resultado_Partido):")
mundial_completo.select("nombre", "edad", "IMC", "Categoria_Edad", "Resultado_Partido").show(10)


## Parte 8: Agregaciones con Spark SQL

In [ ]:
mundial_completo.createOrReplaceTempView("Mundial")

print("8.1. ¿Cuántos jugadores hay por equipo?")
spark.sql("""
    SELECT nombre_equipo, COUNT(DISTINCT jugador_id) as total_jugadores
    FROM Mundial
    GROUP BY nombre_equipo
    ORDER BY total_jugadores DESC
""").show(5)

print("8.2. Edad promedio, mínima y máxima de los jugadores por posición:")
spark.sql("""
    SELECT posicion,
           ROUND(AVG(edad), 2) as edad_promedio,
           MIN(edad) as edad_min,
           MAX(edad) as edad_max
    FROM (SELECT DISTINCT jugador_id, posicion, edad FROM Mundial)
    GROUP BY posicion
""").show()

print("8.3. Altura promedio, mínima y máxima agrupado por confederación:")
spark.sql("""
    SELECT confederacion,
           ROUND(AVG(altura), 2) as altura_promedio,
           MIN(altura) as altura_min,
           MAX(altura) as altura_max
    FROM (SELECT DISTINCT jugador_id, confederacion, altura FROM Mundial)
    GROUP BY confederacion
""").show()

print("8.4. ¿Cuántos partidos se jugaron en cada fase del torneo?")
spark.sql("""
    SELECT fase, COUNT(DISTINCT partido_id) as partidos_jugados
    FROM Mundial
    WHERE fase IS NOT NULL
    GROUP BY fase
""").show()

print("8.5. ¿Cuál es el total de goles anotados por cada equipo (como local y visitante)?")
spark.sql("""
    SELECT nombre_equipo,
           SUM(CASE WHEN equipo_id = equipo_local_id THEN goles_local ELSE 0 END) +
           SUM(CASE WHEN equipo_id = equipo_visitante_id THEN goles_visitante ELSE 0 END) as goles_totales
    FROM (SELECT DISTINCT equipo_id, nombre_equipo, partido_id, equipo_local_id, equipo_visitante_id, goles_local, goles_visitante FROM Mundial WHERE partido_id IS NOT NULL)
    GROUP BY nombre_equipo
    ORDER BY goles_totales DESC
""").show(5)

print("8.6. ¿Cuál es el promedio de goles por partido en cada torneo?")
spark.sql("""
    SELECT nombre_torneo, ROUND(AVG(goles_totales), 2) as promedio_goles
    FROM (
        SELECT DISTINCT partido_id, nombre_torneo, (goles_local + goles_visitante) as goles_totales
        FROM Mundial
        WHERE partido_id IS NOT NULL
    )
    GROUP BY nombre_torneo
""").show()

print("8.7. Capacidad promedio de estadios por país:")
spark.sql("""
    SELECT pais_estadio, ROUND(AVG(capacidad), 0) as capacidad_promedio
    FROM (SELECT DISTINCT estadio_id, pais_estadio, capacidad FROM Mundial WHERE estadio_id IS NOT NULL)
    GROUP BY pais_estadio
""").show()

print("8.8. Clasificación de IMC usando CASE WHEN:")
spark.sql("""
    SELECT DISTINCT nombre, apellido, ROUND(IMC, 2) as IMC,
           CASE
               WHEN IMC < 20 THEN 'Bajo peso'
               WHEN IMC BETWEEN 20 AND 25 THEN 'Normal'
               WHEN IMC > 25 THEN 'Sobrepeso'
           END AS Clasificacion_IMC
    FROM Mundial
""").show(8)

print("8.9. Número de victorias, derrotas y empates de cada equipo:")
spark.sql("""
    SELECT nombre_equipo,
           SUM(CASE WHEN (equipo_id = equipo_local_id AND goles_local > goles_visitante) OR 
                         (equipo_id = equipo_visitante_id AND goles_visitante > goles_local) THEN 1 ELSE 0 END) as victorias,
           SUM(CASE WHEN (equipo_id = equipo_local_id AND goles_local < goles_visitante) OR 
                         (equipo_id = equipo_visitante_id AND goles_visitante < goles_local) THEN 1 ELSE 0 END) as derrotas,
           SUM(CASE WHEN goles_local = goles_visitante THEN 1 ELSE 0 END) as empates
    FROM (SELECT DISTINCT equipo_id, nombre_equipo, partido_id, equipo_local_id, equipo_visitante_id, goles_local, goles_visitante FROM Mundial WHERE partido_id IS NOT NULL)
    GROUP BY nombre_equipo
    ORDER BY victorias DESC
""").show(5)

print("8.10. Equipos con promedio de edad > 28 (Uso de HAVING):")
spark.sql("""
    SELECT nombre_equipo, ROUND(AVG(edad), 2) as promedio_edad
    FROM (SELECT DISTINCT jugador_id, nombre_equipo, edad FROM Mundial)
    GROUP BY nombre_equipo
    HAVING AVG(edad) > 28
""").show()


## Parte 9: Transformaciones Avanzadas

In [ ]:
print("9.1, 9.2 y 9.3: Partidos de la fase Final, ordenado por edad descendente:")
df_final = mundial_completo.filter(F.col("fase") == "Final") \
    .select("nombre", "apellido", "posicion", "nombre_equipo", "edad") \
    .dropDuplicates() \
    .orderBy(F.col("edad").desc())

df_final.show(10)

print("\n9.4: Mostrar los 10 jugadores más altos del torneo:")
jugadores_altos = mundial_completo.select("nombre", "apellido", "altura", "nombre_equipo") \
    .dropDuplicates(["nombre", "apellido"]) \
    .orderBy(F.col("altura").desc())
    
jugadores_altos.show(10)
